In [ ]:
import functools
import math
import pandas as pd

from collections import OrderedDict
from password_policy import PCP

# openpyxl are also required to be installed

# Parse Data

In [ ]:
# Read in the data
with pd.ExcelFile('clean_data.xlsx') as xls:
    websites, dedup, policy = pd.read_excel(xls, None, index_col=0).values()

# Parse complex column types
def parse_dict(data):
    if pd.isnull(data):
        return {}
    values = {}
    exec(f'temp = dict({data})', values)
    return values['temp']

policy['alexa_rank'] = policy['alexa_rank'].apply(parse_dict)
policy['quantcast_rank'] = policy['quantcast_rank'].apply(parse_dict)
policy['policy'] = policy['policy'].apply(PCP.loads)
policy['sso'] = policy['sso'].apply(lambda s: eval(s) if not pd.isna(s) else s)

# High-level analysis

In [ ]:
# Analysis on how many websites we had after cleaning up the data
print(f'Total website: {websites["global_rank"].count()}')
print(f'Deduplicated website: {dedup["global_rank"].count()}')
print(f'Deduplicated websites with policy: {dedup[dedup["has_policy"] | (~dedup["sso"].isnull())]["global_rank"].count()}')
print(f'Policy websites: {policy["global_rank"].count()}')

In [ ]:
# Get a sense of how long the password policies are
policy['policy'].apply(lambda x: len(x.dumps())).describe()

In [ ]:
# Overall strength
policy[['machine_strength','human_strength','chinese_strength']].describe()

# Bin by country

In [ ]:
# Convert the country ranks to a list of countries.
global_set = set(['Global'])
global_cutoff = 1

def calculate_countries(row, cutoff):
    country_list = (set(row['alexa_rank']) | set(row['quantcast_rank'])) - global_set
    if len(country_list) == 0 or len(country_list) > cutoff:
        return global_set
    else:
        return country_list

policy['countries'] = policy.apply(lambda r: calculate_countries(r, global_cutoff), axis=1)
policy['country'] = policy.apply(lambda r: list(calculate_countries(r, 1))[0], axis=1)

# Create bins and associated filtersG
representative_countries = ['Australia', 'Brazil', 'Germany', 'India', 'Nigeria', 'UK', 'US']
repressive_countries = ['China', 'Iran', 'Russia']
countries = ['Global'] + representative_countries + repressive_countries

country_bins = OrderedDict()
for country in countries:
    country_bins[country] = policy['countries'].apply(lambda x: country in x)

In [ ]:
# Print out sizes
for country, test in country_bins.items():
    print(f'{country}: {test.sum()}')

In [ ]:
policy['country'].value_counts()

In [ ]:
# Strength by country
pd.DataFrame({name: policy[test]['machine_strength'] for name, test in country_bins.items()}).describe()

In [ ]:
pd.DataFrame({name: policy[test]['human_strength'] for name, test in country_bins.items()}).describe()

In [ ]:
pd.DataFrame({name: policy[test]['chinese_strength'] for name, test in country_bins.items()}).describe()

# Bin by popularity

In [ ]:
popularities = [
    ('Top 10',   (1,     10)),
    ('Top 50',   (11,    50)),
    ('Top 100',  (51,    100)),
    ('Top 500',  (101,   500)),
    ('Top 1000', (501,   1000)),
    ('Top 5000', (1001,  5000)),
    ('5000+',    (5000,  math.inf))
]

popularity_bins = OrderedDict()
for name, (min_value, max_value) in popularities:
    popularity_bins[name] = policy['global_rank'].apply(lambda x: min_value <= x <= max_value)

In [ ]:
# Print out sizes
for popularity, test in popularity_bins.items():
    print(f'{popularity}: {test.sum()}')

In [ ]:
policy['popularity'] = policy['global_rank'].apply(lambda r: next(name for name, (min_value, max_value) in popularities if min_value <= r <= max_value))

In [ ]:
# Strength by popularity
pd.DataFrame({name: policy[test]['machine_strength'] for name, test in popularity_bins.items()}).describe()

In [ ]:
pd.DataFrame({name: policy[test]['human_strength'] for name, test in popularity_bins.items()}).describe()

In [ ]:
pd.DataFrame({name: policy[test]['chinese_strength'] for name, test in popularity_bins.items()}).describe()

# Bin by category

In [ ]:
# Identify categories with sufficient support to be binned
category_cutoff = 10
included_categories = [category for category, count in policy['category'].value_counts().items() if count >= category_cutoff]
mapped_categories = policy['category'].apply(lambda x: x.capitalize() if x in included_categories else 'Other')

# Create filters for each category
category_bins = OrderedDict()
for category in sorted(mapped_categories.unique()):
    category_bins[category] = mapped_categories.apply(lambda x: category == x)
category_bins['Other'] = category_bins.pop('Other') # Ensures other is at the end

In [ ]:
# Print out sizes
for category, test in category_bins.items():
    print(f'{category}: {test.sum()}')

In [ ]:
policy['use_case'] = mapped_categories

In [ ]:
# Strength by category
pd.DataFrame({name: policy[test]['machine_strength'] for name, test in category_bins.items()}).describe()

In [ ]:
# Strength by category
pd.DataFrame({name: policy[test]['human_strength'] for name, test in category_bins.items()}).describe()

In [ ]:
# Strength by category
pd.DataFrame({name: policy[test]['chinese_strength'] for name, test in category_bins.items()}).describe()